1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



# 🟩 Grupo B — Variables de contenido multimedia

**Variables:**
- `num_imgs`
- `num_videos`

**👉 Aquí evaluamos:**
- si el artículo es visual o no  
- distribución muy sesgada (la mayoría tiene pocos videos/imágenes)

In [ ]:
# ==========================================
# GRUPO B — VARIABLES MULTIMEDIA
# num_imgs, num_videos
# ==========================================

multimedia_vars = ["num_imgs", "num_videos"]

df[multimedia_vars].head()


1. Descripción estadística

In [ ]:
df[multimedia_vars].describe().T


Distribución (histogramas con KDE)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in multimedia_vars:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], bins=40, kde=True)
    plt.title(f"Distribución de {col}")
    plt.show()


3. Boxplots (detección de outliers visuales)

In [ ]:
for col in multimedia_vars:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Outliers en {col}")
    plt.show()


4. Relación con la variable objetivo (shares y log_shares)

crear la variable log-transformada:

In [ ]:
df["log_shares"] = np.log1p(df["shares"])


Correlación con shares:

In [ ]:
df[multimedia_vars + ["shares"]].corr()["shares"].sort_values(ascending=False)


Correlación con log_shares

In [ ]:
df[multimedia_vars + ["log_shares"]].corr()["log_shares"].sort_values(ascending=False)


5. Gráficas de dispersión vs viralidad

In [ ]:
for col in multimedia_vars:
    plt.figure(figsize=(8,4))
    sns.scatterplot(x=df[col], y=df["log_shares"], alpha=0.3)
    plt.title(f"{col} vs log_shares")
    plt.show()


6. Insight rápido (versión automática)

In [ ]:
for col in multimedia_vars:
    print(f"==== {col} ====")
    print(f"Media: {df[col].mean():.2f}")
    print(f"Mediana: {df[col].median():.2f}")
    print(f"Máximo: {df[col].max()}")
    print(f"Valores cero: {df[col].isin([0]).mean()*100:.2f}%")
    print()


Análisis VIF (Grupo B — Multimedia)

In [ ]:
# ==========================================
# Análisis VIF — Grupo B (num_imgs, num_videos)
# ==========================================

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# Variables del grupo B
multimedia_vars = ["num_imgs", "num_videos"]

# Crear dataframe solo con esas variables
X = df[multimedia_vars].copy()

# Agregar constante para el análisis VIF
X["intercept"] = 1

# Calcular VIF
vif_data = pd.DataFrame()
vif_data["variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

vif_data


# 🟩 Conclusiones del Grupo B — Variables Multimedia (`num_imgs`, `num_videos`)

## 📌 1. Distribuciones altamente sesgadas
Las variables `num_imgs` y `num_videos` presentan distribuciones muy sesgadas hacia la derecha.

- `num_imgs`: mediana = 1, promedio = 4.92
- `num_videos`: mediana = 0, promedio = 1.33

👉 La mayoría de artículos incluyen **muy pocas imágenes o videos**, mientras unos pocos casos extremos contienen valores muy altos (hasta 61 imágenes y 26 videos).

**Implicación:**  
Se recomienda considerar transformaciones logarítmicas para modelos lineales y/o winsorización de outliers.

---

## 📌 2. Alto porcentaje de valores cero
- `num_videos`: **63.4% ceros**
- `num_imgs`: **17.3% ceros**

👉 Los artículos rara vez contienen videos, y en general no son muy visuales.

**Implicación:**  
Puede ser útil crear variables binarias como `has_imgs` o `has_videos`.

---

## 📌 3. Correlación débil con la viralidad (`shares` y `log_shares`)
Correlaciones observadas:

### Con `shares`:
- `num_imgs`: 0.057  
- `num_videos`: 0.028  

### Con `log_shares`:
- `num_imgs`: 0.076  
- `num_videos`: 0.026  

👉 Ambas variables muestran **baja correlación** con la viralidad del artículo.

**Conclusión:**  
El contenido multimedia **no es un factor determinante** para predecir cuántas veces será compartido un artículo.

---

## 📌 4. Gráficas de dispersión confirman baja relación
Las gráficas `num_imgs vs log_shares` y `num_videos vs log_shares` muestran una nube de puntos sin patrón.

👉 **No existe una tendencia clara entre multimedia y viralidad.**

---

## 📌 5. Sin multicolinealidad (VIF ≈ 1.0)
El análisis VIF arroja:

| Variable     | VIF    |
|--------------|--------|
| num_imgs     | 1.003  |
| num_videos   | 1.003  |

👉 No existe correlación interna significativa entre estas variables.

**Implicación:**  
Ambas variables pueden mantenerse sin riesgo de redundancia.

---

# 🎯 Conclusión general del Grupo B
El análisis muestra que:

- Las variables multimedia están **muy sesgadas**  
- Contienen **muchos ceros**  
- Tienen **baja correlación con la viralidad**  
- No presentan **multicolinealidad**  
- Son variables de **baja señal**, pero pueden permanecer en el modelo

---

# 🛠 Recomendaciones para Feature Engineering — Grupo B (Multimedia)

# 1. Crear variables binarias
df["has_imgs"] = (df["num_imgs"] > 0).astype(int)
df["has_videos"] = (df["num_videos"] > 0).astype(int)

# 2. Crear un índice de “contenido visual”
df["visual_score"] = df["num_imgs"] + 2 * df["num_videos"]

# 3. Transformaciones opcionales (útiles para modelos lineales)
df["log_num_imgs"] = np.log1p(df["num_imgs"])
df["log_num_videos"] = np.log1p(df["num_videos"])


# 🟩 **Conclusiones Finales — Grupo B (Variables Multimedia)**

## 📌 1. Distribuciones altamente sesgadas
Las variables `num_imgs` y `num_videos` presentan colas largas:

- `num_imgs`: mediana = 1, promedio = 4.92, máximo = 61  
- `num_videos`: mediana = 0, promedio = 1.33, máximo = 26  

➡️ La mayoría de artículos tiene muy pocas imágenes o videos, pero algunos casos extremos elevan la media.

**Implicación:**  
Transformaciones logarítmicas (`log1p`) o winsorización pueden estabilizar modelos lineales.

---

## 📌 2. Alto porcentaje de valores cero
- `num_videos`: **63.4%** son ceros  
- `num_imgs`: **17.3%** son ceros  

➡️ Indica que los artículos rara vez contienen multimedia.

**Implicación:**  
Crear **variables binarias** puede capturar mejor la señal:  
`has_imgs`, `has_videos`.

---

## 📌 3. Correlación débil con viralidad
Correlaciones con shares y log_shares son muy bajas:

| Variable     | Corr. shares | Corr. log_shares |
|--------------|--------------|------------------|
| num_imgs     | 0.057        | 0.076            |
| num_videos   | 0.028        | 0.026            |

➡️ El contenido visual **no explica la viralidad**.

---

## 📌 4. Gráficas de dispersión confirman relación débil
Las gráficas `num_imgs vs log_shares` y `num_videos vs log_shares` muestran nubes de puntos sin patrón.

➡️ No hay tendencia clara.

---

## 📌 5. Sin multicolinealidad (VIF ≈ 1.0)
El VIF confirma que las variables no son redundantes:

- VIF(num_imgs) ≈ 1.00  
- VIF(num_videos) ≈ 1.00  

➡️ Ambas pueden incluirse sin riesgo.

---

# 🎯 **Conclusión General — Grupo B**
- Las variables multimedia aportan **señal muy débil**.  
- Son útiles como **features auxiliares**, no principales.  
- Deben mantenerse, pero transformadas o como binarias.

---

# 🛠 **Feature Engineering Recomendado — Grupo B**

## ✔ 1. Variables binarias
```python
df["has_imgs"] = (df["num_imgs"] > 0).astype(int)
df["has_videos"] = (df["num_videos"] > 0).astype(int)
